In [1]:

from langchain_chroma import Chroma 
from langchain_community.vectorstores import FAISS
from langchain_community.docstore.in_memory import InMemoryDocstore
from langchain_community.docstore.document import Document
from pinecone import Pinecone, ServerlessSpec
from langchain_openai import OpenAIEmbeddings
from langchain_pinecone import PineconeVectorStore

import os ,faiss
from dotenv import load_dotenv
# Use override=True to force-reload from .env over cached OS variables
load_dotenv(override=True)

key = os.getenv("OPENAI_API_KEY")
print(f"Length: {len(key) if key else 0}")
print(f"Starts with: {repr(key[:12]) if key else None}")
print(f"Ends with: {repr(key[-6:]) if key else None}")

C:\Users\RISHABH\AppData\Local\Temp\ipykernel_24924\3071228548.py:2: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.vectorstores import FAISS


Length: 164
Starts with: 'sk-proj-PqZV'
Ends with: 'HqkfMA'


d:\codes\Lab\GenAI_\genaiu\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
doc1 = Document(
    page_content="INTJ is known as the Architect. INTJs are strategic, independent, and analytical thinkers who enjoy planning for the future and solving complex problems. They value competence, efficiency, and long-term goals.",
    metadata={"MBTI_type": "INTJ"}
)

doc2 = Document(
    page_content="INFP is known as the Mediator. INFPs are imaginative, empathetic, and idealistic individuals who value authenticity and personal meaning. They often express themselves through creativity and seek harmony between their actions and beliefs.",
    metadata={"MBTI_type": "INFP"}
)

doc3 = Document(
    page_content="ENTP is known as the Debater. ENTPs are curious, inventive, and energetic individuals who enjoy exploring new ideas, questioning assumptions, and discussing different possibilities. They are often adaptable and skilled at brainstorming solutions.",
    metadata={"MBTI_type": "ENTP"}
)

doc4 = Document(
    page_content="ISFJ is known as the Defender. ISFJs are considerate, dependable, and practical individuals who value responsibility and helping others. They often pay attention to details, appreciate stability, and work hard to support the people they care about.",
    metadata={"MBTI_type": "ISFJ"}
)

doc5 = Document(
    page_content="ENFJ is known as the Protagonist. ENFJs are sociable, empathetic, and organized individuals who often enjoy motivating others and building strong relationships. They tend to value cooperation, personal growth, and shared goals.",
    metadata={"MBTI_type": "ENFJ"}
)

# TABLE OF CONTENTS 
1)  Chroma DB 
2)  FAISS
3)  Pinecone  (API KEY based )

![Chroma vs FAISS vs Pinecone](assets/vector_store_comparison.png)

## 1)  Chroma DB 

In [3]:

embeddings = OpenAIEmbeddings(
    model="text-embedding-3-small",
    api_key=key
)

chroma_vector_store = Chroma(
    embedding_function=embeddings,
    persist_directory="./db",
    collection_name="sample"
)

chroma_vector_store.add_documents(
    documents=[doc1, doc2, doc3, doc4, doc5]
)

['7f8b8dc9-fbd2-4212-9e26-4dc822901e3a',
 'fbad403d-f6b6-456c-bc5c-76b96e8e244a',
 '6cfc3537-88a1-4862-adf6-fa69fe8fae64',
 '1f7e8175-e9ea-4026-af46-38af02436823',
 '866a802d-10d9-4390-ac1b-ae1a485c2436']

In [4]:
result = chroma_vector_store.similarity_search(
    query="who is most analytical MBTI type ?",
    k=2
)

# vectorstore.similarity_search_with_score # with score 
print(result)


[Document(id='9a30edec-836f-4057-8d70-cc99f7f61710', metadata={'MBTI_type': 'INTJ'}, page_content='INTJ is known as the Architect. INTJs are strategic, independent, and analytical thinkers who enjoy planning for the future and solving complex problems. They value competence, efficiency, and long-term goals.'), Document(id='7f8b8dc9-fbd2-4212-9e26-4dc822901e3a', metadata={'MBTI_type': 'INTJ'}, page_content='INTJ is known as the Architect. INTJs are strategic, independent, and analytical thinkers who enjoy planning for the future and solving complex problems. They value competence, efficiency, and long-term goals.')]


## 2)  FAISS 

In [5]:
faiss_vectorstore = FAISS.from_documents(
    documents=[doc1, doc2, doc3, doc4, doc5],
    embedding=embeddings
)

# --- Approach B: Empty init + add_documents (Chroma style) ---
# Get vector dimension dynamically (e.g., 1536 for OpenAI)
embedding_dim = len(embeddings.embed_query("hello world"))
index = faiss.IndexFlatL2(embedding_dim)

faiss_vectorstore = FAISS(
    embedding_function=embeddings,
    index=index,
    docstore=InMemoryDocstore(),
    index_to_docstore_id={}
)
faiss_vectorstore.add_documents(documents=[doc1, doc2, doc3, doc4, doc5])

# --- Saving and Loading (Replaces Chroma's persist_directory) ---
# 1. Save to disk manually
faiss_vectorstore.save_local(folder_path="./faiss_db", index_name="sample")

# 2. Load it back later
loaded_vectorstore = FAISS.load_local(
    folder_path="./faiss_db",
    embeddings=embeddings,
    index_name="sample",
    allow_dangerous_deserialization=True  # Required by LangChain for local pickle files
)

In [6]:
result =  loaded_vectorstore.similarity_search(
    query="Which MBTI type is most analytical ?",
    k=2
)
print(result)

[Document(id='95a46653-8fda-4c33-937a-7eed5cad8af9', metadata={'MBTI_type': 'INTJ'}, page_content='INTJ is known as the Architect. INTJs are strategic, independent, and analytical thinkers who enjoy planning for the future and solving complex problems. They value competence, efficiency, and long-term goals.'), Document(id='8fe90621-674b-4dc1-925b-d14e4cbf58f7', metadata={'MBTI_type': 'ENTP'}, page_content='ENTP is known as the Debater. ENTPs are curious, inventive, and energetic individuals who enjoy exploring new ideas, questioning assumptions, and discussing different possibilities. They are often adaptable and skilled at brainstorming solutions.')]


## 3) Pinecone 

In [7]:
# 1. Initialize Embeddings
embeddings = OpenAIEmbeddings(model="text-embedding-3-small")

# 2. Connect to Pinecone and create index if it doesn't exist
pc = Pinecone(api_key=os.getenv("PINECONE_API_KEY"))
index_name = "sample-index"  # Must be lowercase alphanumeric and hyphens only

if not pc.has_index(index_name):
    pc.create_index(
        name=index_name,
        dimension=1536,  # Must match your OpenAI embedding model's dimension
        metric="cosine",
        spec=ServerlessSpec(cloud="aws", region="us-east-1")
    )

index = pc.Index(index_name)

# 3. Initialize VectorStore (namespace acts like Chroma's collection_name)
vectorstore = PineconeVectorStore(
    index=index,
    embedding=embeddings,
    namespace="sample"
)

# 4. Add documents just like Chroma
vectorstore.add_documents(documents=[doc1, doc2, doc3, doc4, doc5])

['780a88e6-2dfe-4720-8e0f-785ff732193c',
 '02f237fd-176a-4799-94e8-91f2b2c8bf5f',
 '2f5e69c0-9c03-4928-9680-b66806b5303e',
 '7f74d9c3-84ca-4e45-aa20-379a284f3ec7',
 '9698d824-ad9b-405d-b73d-231c041b5823']

In [8]:
print(vectorstore.similarity_search_with_score(
    query="most coolest MBTI type  ?",
    k = 2
))

[(Document(id='780a88e6-2dfe-4720-8e0f-785ff732193c', metadata={'MBTI_type': 'INTJ'}, page_content='INTJ is known as the Architect. INTJs are strategic, independent, and analytical thinkers who enjoy planning for the future and solving complex problems. They value competence, efficiency, and long-term goals.'), 0.545895398), (Document(id='2ae01ba3-7817-4b99-bccf-097bae93bdbd', metadata={'MBTI_type': 'INTJ'}, page_content='INTJ is known as the Architect. INTJs are strategic, independent, and analytical thinkers who enjoy planning for the future and solving complex problems. They value competence, efficiency, and long-term goals.'), 0.545023)]


#### COMBINED RESULT 

In [9]:


query = "which MBTI type is good at debating  ? "

for i in [chroma_vector_store,loaded_vectorstore,vectorstore]:
    result =  i.similarity_search(
       query =  query, k=2 
    )
    
    print(result)
    
    

[Document(id='ee64a8b4-46bb-427f-9abd-573f3180be68', metadata={'MBTI_type': 'ENTP'}, page_content='ENTP is known as the Debater. ENTPs are curious, inventive, and energetic individuals who enjoy exploring new ideas, questioning assumptions, and discussing different possibilities. They are often adaptable and skilled at brainstorming solutions.'), Document(id='6cfc3537-88a1-4862-adf6-fa69fe8fae64', metadata={'MBTI_type': 'ENTP'}, page_content='ENTP is known as the Debater. ENTPs are curious, inventive, and energetic individuals who enjoy exploring new ideas, questioning assumptions, and discussing different possibilities. They are often adaptable and skilled at brainstorming solutions.')]
[Document(id='8fe90621-674b-4dc1-925b-d14e4cbf58f7', metadata={'MBTI_type': 'ENTP'}, page_content='ENTP is known as the Debater. ENTPs are curious, inventive, and energetic individuals who enjoy exploring new ideas, questioning assumptions, and discussing different possibilities. They are often adaptab